## Model parameters

- `temperature` sets how much the answer varies between runs
- `top_p` narrows the pool of words the model may choose from
- `max_tokens` caps the length, and will cut a sentence in half to do it

Runs one prompt at several settings so the difference is visible.

### installation and configuration

In [ ]:
# (setup cell already installs what this notebook needs)

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

### helper function for tests

In [ ]:
%pip install -q langchain-ollama

In [ ]:
from langchain_openai import ChatOpenAI

# Local: use llama.cpp (OpenAI-compatible) instead of Ollama.
# Map Ollama-only parameter names to their OpenAI/llama.cpp equivalents.
_OLLAMA_TO_OPENAI = {"num_predict": "max_tokens"}

def test_generation(param_name, values, prompt="Invent a short fairy tale about a cat and a dog."):
    for v in values:
        print("=" * 60)
        print(f"{param_name} = {v}")

        settings = {"temperature": 0.7}      # base default
        key = _OLLAMA_TO_OPENAI.get(param_name, param_name)
        settings[key] = v                    # override the parameter under test

        llm = make_llm()
        text = llm.invoke(prompt).content

        print(f"[{len(text.split())} words]")   # proof the cap is applied
        print(text, "\n")


### Exercise Experiment with temperature 
Run the model with temperature = 0.0, 0.7 and 1.0 \
Compare the diversity of the responses (compare the length and note 1 difference).

In [ ]:
test_generation("temperature", [0.0, 0.7, 1.2])

### Exercise experiment with top_p
Change top_p (0.3 vs 0.95) and briefly comment on the effect on the result.

In [ ]:
test_generation("top_p", [0.2, 0.7, 1.0])

### Exercise experiment with num_predict
1. Set `max_tokens` to a small value (e.g. 32) and a large one (e.g. 256). Check whether the response gets cut off.
2. Add a simple function `truncate_warning(text)` that prints a warning if the model ends a sentence midway.
3. Compare the response time between a small and a large `max_tokens` (use e.g. `time.perf_counter`).

In [ ]:
test_generation("num_predict", [30, 100, 300])

💡 Effect:\
with temperature=0 the responses will be repetitive,\
with temperature=1.2, more wild,\
with a low top_p the model will be conservative,\
with a high top_k, more diverse,\
and max_tokens will decide whether the fairy tale is 2 sentences or a whole page.

### Try temperature 0 twice

- Run the same prompt twice at 0.0, then twice at 1.0
- At 0 the answers should match. That is the setting to use when we are
  testing anything else, because it removes one source of noise